# Week 9 — Decision Tree Deep Dive

This notebook covers the Week 9 milestone: a thorough investigation of the CART Decision Tree on the RAVDESS SER problem.

**Goals:**
- Understand how `max_depth` controls the bias-variance trade-off (overfitting curve)
- Visualize the tree structure so we can see which features drive splits
- Explore pre-pruning via `min_samples_split`
- Compare Gini impurity vs. entropy as split criteria
- Inspect the most-used features as a proxy for feature importance
- Benchmark MiniLearn's Decision Tree against scikit-learn's implementation

The MiniLearn `DecisionTreeClassifier` uses the **CART algorithm** with Gini impurity (default) or entropy, and supports `max_depth` and `min_samples_split` as stopping/pruning controls.

In [ ]:
import sys
import os
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier as SklearnDT

sys.path.insert(0, os.path.abspath('..'))

from minilearn.preprocessing import StandardScaler, train_test_split
from minilearn.classifiers import DecisionTreeClassifier
from minilearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
EMOTION_ORDER = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

## 1. Load Feature Matrix

In [ ]:
df = pd.read_csv('../outputs/features.csv')

LABEL_COLS = ['filename', 'emotion', 'emotion_id', 'actor', 'gender', 'channel']
feature_cols = [c for c in df.columns if c not in LABEL_COLS]
feature_names = feature_cols

X = df[feature_cols].values
y = df['emotion'].values

print(f'Feature matrix : {X.shape}')
print(f'Classes        : {np.unique(y).tolist()}')
print(f'Total samples  : {len(y)}')

## 2. Train / Test Split and Feature Scaling

Same 80/20 split and `StandardScaler` setup used in all other notebooks.
The scaler is **fit only on training data** to prevent data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train_s.shape}  |  Test: {X_test_s.shape}')

## 3. Overfitting Curve — `max_depth` Sweep

The core tension in Decision Trees is **bias vs. variance**:
- A **shallow tree** underfits — it can't model complex decision boundaries, so both train and test accuracy are low.
- A **deep tree** overfits — it memorises training noise, so train accuracy climbs to ~100% while test accuracy plateaus then falls.

The sweet spot is wherever test accuracy peaks.

> **Note:** MiniLearn's DT evaluates up to `max_thresholds` candidate split points per feature. We use `max_thresholds=16` here to keep the sweep tractable; the final model uses 64.

In [ ]:
DEPTHS = [1, 2, 3, 5, 7, 10, 15, 20]

train_accs, test_accs, sweep_times = [], [], []

for d in DEPTHS:
    clf = DecisionTreeClassifier(max_depth=d, max_thresholds=16)
    t0 = time.time()
    clf.fit(X_train_s, y_train)
    elapsed = time.time() - t0

    tr_acc = accuracy_score(y_train, clf.predict(X_train_s))
    te_acc = accuracy_score(y_test,  clf.predict(X_test_s))

    train_accs.append(tr_acc)
    test_accs.append(te_acc)
    sweep_times.append(elapsed)
    print(f'depth={d:2d}  train={tr_acc:.3f}  test={te_acc:.3f}  time={elapsed:.1f}s')

best_idx = int(np.argmax(test_accs))
best_depth = DEPTHS[best_idx]
print(f'\nBest test accuracy {test_accs[best_idx]:.3f} at max_depth={best_depth}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(DEPTHS, train_accs, 'o-', label='Train accuracy', color='steelblue', linewidth=2)
ax.plot(DEPTHS, test_accs,  's-', label='Test accuracy',  color='tomato',    linewidth=2)
ax.axvline(best_depth, color='gray', linestyle='--', alpha=0.8,
           label=f'Best depth = {best_depth}  (test acc = {test_accs[best_idx]:.3f})')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: Depth vs. Accuracy (Overfitting Curve)')
ax.set_xticks(DEPTHS)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/dt_depth_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/dt_depth_sweep.png')

### What the curve shows

Training accuracy climbs monotonically with depth — a deeper tree can always split further until it memorises the training set. Test accuracy, however, peaks at a much shallower depth and then plateaus or drops as the model starts fitting noise rather than signal. This gap between train and test accuracy is the classic signature of **overfitting**.

Limiting `max_depth` is a form of **pre-pruning**: we stop growing the tree before it becomes too complex, even if the training data supports further splits.

## 4. Tree Visualization

We train the best-depth tree with full `max_thresholds=64` and then print the first few levels. Each internal node shows which feature is being tested and the threshold value. Leaf nodes show the predicted class label.

Because MiniLearn uses raw feature indices internally, we map them back to human-readable feature names.

In [ ]:
clf_best = DecisionTreeClassifier(max_depth=best_depth, max_thresholds=64)
clf_best.fit(X_train_s, y_train)

y_pred_best = clf_best.predict(X_test_s)
print(f'Best DT (depth={best_depth})  '
      f'acc={accuracy_score(y_test, y_pred_best):.3f}  '
      f'macro-F1={f1_score(y_test, y_pred_best, average="macro"):.3f}')

In [ ]:
def print_tree(node, feature_names, depth=0, max_print_depth=4):
    """Recursively print the tree structure, capped at max_print_depth."""
    indent = '    ' * depth
    if node.value is not None:
        print(f'{indent}[LEAF] → {node.value}')
        return
    if depth >= max_print_depth:
        print(f'{indent}... (subtree continues)')
        return
    name = (feature_names[node.feature]
            if node.feature < len(feature_names)
            else f'feat_{node.feature}')
    print(f'{indent}[d={depth}] {name} <= {node.threshold:.4f}')
    print(f'{indent}  ├─ (≤) :')
    print_tree(node.left,  feature_names, depth + 1, max_print_depth)
    print(f'{indent}  └─ (>) :')
    print_tree(node.right, feature_names, depth + 1, max_print_depth)


print(f'Tree structure (top 4 levels shown) — max_depth={best_depth}\n')
print_tree(clf_best.root_, feature_names, max_print_depth=4)

In [ ]:
def tree_stats(node, depth=0):
    """Return (actual_depth, n_leaves, n_internal) by traversing the tree."""
    if node.value is not None:
        return depth, 1, 0
    ld, ll, li = tree_stats(node.left,  depth + 1)
    rd, rl, ri = tree_stats(node.right, depth + 1)
    return max(ld, rd), ll + rl, li + ri + 1


actual_depth, n_leaves, n_internal = tree_stats(clf_best.root_)
print(f'Actual depth reached : {actual_depth}  (max_depth limit = {best_depth})')
print(f'Leaf nodes           : {n_leaves}')
print(f'Internal split nodes : {n_internal}')
print(f'Total nodes          : {n_leaves + n_internal}')

## 5. Feature Usage (Proxy for Importance)

A simple way to gauge which features the tree relies on most is to count how many split nodes use each feature. Features that appear at many splits are heavily relied upon for discrimination.

> A more principled measure (sklearn's `feature_importances_`) weights each split by the Gini reduction it achieves, multiplied by the fraction of training samples that reach that node. Our count-based proxy is coarser but still informative.

In [ ]:
def feature_usage(node, counts=None):
    if counts is None:
        counts = Counter()
    if node.value is not None:
        return counts
    counts[node.feature] += 1
    feature_usage(node.left,  counts)
    feature_usage(node.right, counts)
    return counts


usage = feature_usage(clf_best.root_)
TOP_N = 15
top_items = usage.most_common(TOP_N)
feat_indices, feat_counts = zip(*top_items)
feat_labels = [feature_names[i] for i in feat_indices]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(range(TOP_N), feat_counts[::-1], color='steelblue')
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(feat_labels[::-1], fontsize=9)
ax.set_xlabel('Number of split nodes using this feature')
ax.set_title(f'Top {TOP_N} Most-Used Features — Decision Tree (depth={best_depth})')
plt.tight_layout()
plt.savefig('../outputs/dt_feature_usage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/dt_feature_usage.png')

## 6. Split Criterion — Gini vs. Entropy

CART uses **Gini impurity** by default:
$$\text{Gini}(S) = 1 - \sum_{k} p_k^2$$

An alternative is **entropy** (information gain):
$$H(S) = -\sum_{k} p_k \log_2 p_k$$

Both measure node impurity; in practice they produce very similar trees and accuracy numbers. Entropy is slightly more expensive to compute (log vs. square).

In [ ]:
criterion_results = {}

for criterion in ['gini', 'entropy']:
    clf = DecisionTreeClassifier(
        max_depth=best_depth, criterion=criterion, max_thresholds=32
    )
    t0 = time.time()
    clf.fit(X_train_s, y_train)
    elapsed = time.time() - t0

    y_pred_c = clf.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred_c)
    f1  = f1_score(y_test, y_pred_c, average='macro')
    criterion_results[criterion] = (acc, f1)
    print(f'criterion={criterion:7s}  test_acc={acc:.3f}  macro_F1={f1:.3f}  train_time={elapsed:.1f}s')

gini_acc, gini_f1 = criterion_results['gini']
entr_acc, entr_f1 = criterion_results['entropy']
print(f'\nΔ accuracy : {abs(gini_acc - entr_acc):.4f}')
print(f'Δ macro-F1 : {abs(gini_f1 - entr_f1):.4f}')

## 7. Pre-Pruning via `min_samples_split`

Besides limiting depth, we can prevent over-splitting by requiring a minimum number of samples before a node is allowed to split. This is called **pre-pruning**.

- **Low** `min_samples_split` → tree grows aggressively, fits noise.
- **High** `min_samples_split` → tree is shallower and coarser; may underfit.

We hold `max_depth` fixed at the best value from Section 3 and vary `min_samples_split`.

In [ ]:
MIN_SAMPLES_LIST = [2, 5, 10, 20, 50, 100]
prune_train_accs, prune_test_accs = [], []

for ms in MIN_SAMPLES_LIST:
    clf = DecisionTreeClassifier(
        max_depth=best_depth, min_samples_split=ms, max_thresholds=32
    )
    clf.fit(X_train_s, y_train)
    prune_train_accs.append(accuracy_score(y_train, clf.predict(X_train_s)))
    prune_test_accs.append(accuracy_score(y_test,  clf.predict(X_test_s)))
    print(f'min_samples_split={ms:3d}  '
          f'train={prune_train_accs[-1]:.3f}  '
          f'test={prune_test_accs[-1]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(MIN_SAMPLES_LIST, prune_train_accs, 'o-', label='Train accuracy',
        color='steelblue', linewidth=2)
ax.plot(MIN_SAMPLES_LIST, prune_test_accs,  's-', label='Test accuracy',
        color='tomato',    linewidth=2)
ax.set_xlabel('min_samples_split')
ax.set_ylabel('Accuracy')
ax.set_title(f'Pre-Pruning Effect (max_depth={best_depth})')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/dt_pruning.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/dt_pruning.png')

## 8. Final Model — Full Evaluation

Using the best `max_depth` found in the depth sweep, we report all standard metrics: accuracy, macro-averaged precision/recall/F1, per-class breakdown, and a normalised confusion matrix.

In [ ]:
acc_best = accuracy_score(y_test, y_pred_best)
f1_best  = f1_score(y_test, y_pred_best, average='macro')

print(f'Final Decision Tree  max_depth={best_depth}')
print(f'  Test Accuracy : {acc_best:.3f}')
print(f'  Macro F1      : {f1_best:.3f}')
print()
print(classification_report(
    y_test, y_pred_best,
    labels=EMOTION_ORDER,
    target_names=EMOTION_ORDER
))

In [ ]:
cm = confusion_matrix(y_test, y_pred_best, labels=EMOTION_ORDER)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=EMOTION_ORDER, yticklabels=EMOTION_ORDER,
    ax=ax, vmin=0, vmax=1, linewidths=0.4
)
ax.set_title(f'Decision Tree (depth={best_depth}) — Normalised Confusion Matrix\n'
             f'acc={acc_best:.3f}  macro-F1={f1_best:.3f}')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../outputs/dt_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/dt_confusion_matrix.png')

## 9. MiniLearn vs. scikit-learn

We compare our from-scratch CART implementation against scikit-learn's `DecisionTreeClassifier` using the same depth and Gini criterion. Any gap in accuracy reflects implementation differences (threshold sampling strategy, tie-breaking, etc.).

In [ ]:
# sklearn DT
sk_clf = SklearnDT(max_depth=best_depth, criterion='gini', random_state=RANDOM_STATE)
t0 = time.time()
sk_clf.fit(X_train_s, y_train)
sk_train_time = time.time() - t0
sk_pred = sk_clf.predict(X_test_s)

sk_acc = accuracy_score(y_test, sk_pred)
sk_f1  = f1_score(y_test, sk_pred, average='macro')

# MiniLearn DT (already trained above as clf_best)
ml_acc = acc_best
ml_f1  = f1_best

comparison = pd.DataFrame([
    {'Implementation': 'MiniLearn DT',  'Test Accuracy': ml_acc, 'Macro F1': ml_f1},
    {'Implementation': 'scikit-learn DT', 'Test Accuracy': sk_acc, 'Macro F1': sk_f1},
    {'Implementation': 'Difference (|Δ|)',
     'Test Accuracy': abs(ml_acc - sk_acc),
     'Macro F1': abs(ml_f1 - sk_f1)},
])
comparison = comparison.set_index('Implementation')
print(comparison.to_string(float_format='{:.4f}'.format))
comparison

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (name, y_p) in zip(axes, [
    (f'MiniLearn DT (depth={best_depth})', y_pred_best),
    (f'sklearn DT  (depth={best_depth})', sk_pred),
]):
    cm_i = confusion_matrix(y_test, y_p, labels=EMOTION_ORDER)
    cm_n = cm_i.astype(float) / cm_i.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_n, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=EMOTION_ORDER, yticklabels=EMOTION_ORDER,
        ax=ax, vmin=0, vmax=1, linewidths=0.3
    )
    ax_acc = accuracy_score(y_test, y_p)
    ax.set_title(f'{name}\nacc={ax_acc:.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('MiniLearn vs. scikit-learn — Decision Tree Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/dt_minilearn_vs_sklearn.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to outputs/dt_minilearn_vs_sklearn.png')

## 10. Discussion

### Overfitting and depth
The depth sweep clearly shows the bias-variance trade-off in action. Training accuracy rises monotonically with depth because a deep enough tree can simply memorise every training sample (each leaf holds one unique path). Test accuracy peaks at a much shallower depth — beyond that point, extra splits are fitting noise specific to the training set rather than the underlying emotion patterns. This is **overfitting**.

### Pre-pruning strategies
Both `max_depth` and `min_samples_split` act as pre-pruning controls — they prevent the tree from growing too complex in the first place. `max_depth` is the blunter tool: it hard-stops at a fixed level regardless of how much information remains. `min_samples_split` is more adaptive: nodes with fewer than the threshold samples are left as leaves, which naturally prevents spurious splits on small subgroups. In practice, combining both gives the most control.

### Gini vs. entropy
The two criteria produce nearly identical results on this dataset. Gini is preferred in practice because it avoids the log computation and is slightly faster, especially relevant in our pure-Python MiniLearn implementation.

### MiniLearn vs. scikit-learn
Any accuracy gap between MiniLearn and sklearn stems from one key design difference: MiniLearn samples a fixed number of candidate thresholds per feature (percentile-based), whereas sklearn's C-extension evaluates every unique value. This means MiniLearn may miss the globally optimal split threshold, slightly reducing accuracy. The trade-off is training speed in pure Python.

### Decision Tree in the SER context
Even at its best depth, the Decision Tree underperforms Logistic Regression (73% vs ~49%) on RAVDESS. Audio emotion features are continuous and high-dimensional (194 features), which plays to the strengths of linear and kernel-based models. Decision trees partition the feature space with axis-aligned cuts, which struggle to capture the correlated, overlapping distributions typical of speech emotion features. This motivates the move to ensemble methods (Week 10), where many trees together compensate for the weakness of any single one.